In [1]:
# =============================================================================
# Export CARIACO scenario data to CSV for downstream Python analysis
# =============================================================================

library(tidyverse)

# Source depth profile functions (also loads scenario_classification.R)
source("depth_profile_data.R")

# -----------------------------------------------------------------------------
# Config — change depth_mode here and re-run to produce different CSV versions
# -----------------------------------------------------------------------------
depth_mode   <- "dynamic"  # "fixed" | "scenario" | "dynamic" | "observed"
fixed_depth  <- 50
output_dir   <- "../processed"
output_file  <- sprintf("cariaco_monthly_euphotic_%s.csv", depth_mode)
output_path  <- file.path(output_dir, output_file)

# -----------------------------------------------------------------------------
# Load data + fit proxy model (isotherm + Chl, best-fitting variant)
# -----------------------------------------------------------------------------
profile_data <- load_profile_data()

proxy_model <- fit_euphotic_proxy(
  profile_data$scenario,
  niskin_profiles = profile_data$niskin,
  model_type      = "isotherm_chl"
)

# -----------------------------------------------------------------------------
# Build full monthly scenario dataframe
# -----------------------------------------------------------------------------
full_data <- get_full_scenario_data(
  profile_data,
  depth_mode   = depth_mode,
  fixed_depth  = fixed_depth,
  proxy_model  = proxy_model
)

# -----------------------------------------------------------------------------
# Write CSV
# -----------------------------------------------------------------------------
write_csv(full_data, output_path)

cat(sprintf("\n=== Wrote %d rows to %s ===\n", nrow(full_data), output_path))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Lade nötiges Paket: gsw



Loaded HPLC profiles: 176 dates, depths 0-200m
Interpolating Niskin data (this may take a moment)...
  Interpolating NO3_merged...
  Interpolating Chlorophyll...
  Interpolating Phaeopigments...
  Interpolating PrimaryProductivity...
  Interpolating PN_ug_L...
  Interpolating Temperature...
Saved Niskin profiles to cache: ../processed/Niskin_depth_profiles.rds
  Dates: 230, depths 0-200m

=== Date Coverage Summary ===
  Total unique dates:     275
  With HPLC data:         176
  With Niskin data:       230
  With scenario class:    257
  With observed EuZ:      102
Chlorophyll integration (0-100m):
  Dates with sufficient Chl data: 229

Fitting EuZ proxy model (isotherm_chl) on 94 observations...
  Model: EuZ = 81.55 + 0.0983 × Isotherm_21 + -30.4234 × log10(Chl_int)
  Coefficient signs: Isotherm ✓ (expect +), log(Chl) ✓ (expect -)
  R-squared: 0.717 (adj: 0.710)
  RMSE: 6.94 m

Prediction coverage:
  Total dates with Isotherm_21: 229
  With observed EuZ: 94
  With predicted EuZ (iso+c

In [2]:
library(tidyverse)
source("depth_profile_data.R")

# Reload (or use full_data from the previous run)
full_data <- read_csv("../processed/cariaco_monthly_euphotic_dynamic.csv")

# --- 1. Column sanity check ---
new_cols <- c("PP_mmolN_m3_d", "Chl_niskin_mgm3", "Phaeo_niskin_mgm3",
              "Chl_niskin_mmolN", "PhaeoChl_ratio")
cat("New columns present:\n")
print(new_cols %in% names(full_data))

# --- 2. Detailed summary with new blocks ---
summary_out <- summarize_full_scenario_detailed(full_data)

# --- 3. Quick range checks ---
full_data %>%
  select(all_of(new_cols)) %>%
  summary() %>%
  print()

# --- 4. Consistency checks ---
# HPLC TotChlA (mmol N) should be in the same ballpark as Niskin Chl (mmol N)
# when both are available on the same month
full_data %>%
  filter(!is.na(TotChlA_mmolN) & !is.na(Chl_niskin_mmolN)) %>%
  summarize(
    n               = n(),
    TotChlA_HPLC    = mean(TotChlA_mmolN),
    Chl_Niskin      = mean(Chl_niskin_mmolN),
    ratio_Niskin_HPLC = mean(Chl_niskin_mmolN / TotChlA_mmolN, na.rm = TRUE)
  ) %>%
  print()

# PP volumetric vs areal consistency:
# PP_mmolN_m3_d * depth_cutoff should ≈ PP_mgC_m2_d converted to N
# i.e. PP_mgC_m2_d / MW_C * (16/106) ≈ PP_mmolN_m3_d * depth_cutoff
full_data %>%
  filter(!is.na(PP_mmolN_m3_d) & !is.na(PP_mgC_m2_d)) %>%
  mutate(
    pp_areal_N_from_vol   = PP_mmolN_m3_d * depth_cutoff,
    pp_areal_N_from_areal = PP_mgC_m2_d / 12.01 * (16/106),
    rel_diff = (pp_areal_N_from_vol - pp_areal_N_from_areal) / pp_areal_N_from_areal
  ) %>%
  summarize(
    n        = n(),
    max_rel_diff = max(abs(rel_diff)),
    mean_rel_diff = mean(rel_diff)
  ) %>%
  print()

Rows: 255 Columns: 42
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (4): time_month, upwelling, ui, cutoff_source
dbl  (37): year, month, Isotherm_21, sst, temp_50m, euphotic_depth_obs, MLD,...
date  (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


New columns present:
[1] TRUE TRUE TRUE TRUE TRUE

╔══════════════════════════════════════════════════════════════════╗
║           CARIACO SCENARIO DATA SUMMARY                          ║
╚══════════════════════════════════════════════════════════════════╝

=== Data Coverage (n observations per variable) ===
 upwelling_group total_dates phyto_size TotChlA NO3 PON  PP Temperature
         relaxed         141         64      88 106 107 112         119
    unclassified          27          0       0   0   0   0           0
       upwelling          87         31      54  75  76  77          76
 zoo_gt200 zoo_gt500 export_flux Isotherm_21 PP_mmolN Chl_niskin Phaeo_niskin
        97        96          86         141      112        120          120
         3         3           6           0        0          0            0
        51        52          47          87       77         78           78
 PhaeoChl
      120
        0
       78

=== Integration Depth ===
  Mode: not recorded
#